In [121]:
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageFilter, ImageEnhance
import os
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
import random
from concurrent.futures import ThreadPoolExecutor

## chuyển đổi hình ảnh trên tập dữ liệu CIC DDOS 2019

In [122]:

selected_columns = ['tot_fwd_pkts','tot_bwd_pkts','totlen_fwd_pkts','totlen_bwd_pkts',
 'fwd_pkt_len_max','fwd_pkt_len_min','fwd_pkt_len_mean','fwd_pkt_len_std',
 'bwd_pkt_len_max','bwd_pkt_len_min','bwd_pkt_len_mean','bwd_pkt_len_std',
 'flow_byts_s','flow_pkts_s','flow_iat_mean','flow_iat_std','flow_iat_max',
 'flow_iat_min','fwd_iat_tot','fwd_iat_mean','fwd_iat_std','fwd_iat_max',
 'fwd_iat_min','bwd_iat_tot','bwd_iat_mean','bwd_iat_std','bwd_iat_max',
 'bwd_iat_min','fwd_psh_flags','bwd_psh_flags','fwd_urg_flags','bwd_urg_flags',
 'fwd_header_len','bwd_header_len','pkt_len_min','pkt_len_max','pkt_len_mean',
 'pkt_len_std','pkt_len_var','pkt_size_avg','fwd_seg_size_avg','bwd_seg_size_avg',
 'fwd_byts_b_avg','fwd_pkts_b_avg','bwd_byts_b_avg','bwd_pkts_b_avg',
 'subflow_fwd_pkts','subflow_fwd_byts','subflow_bwd_pkts','subflow_bwd_byts',
 'init_fwd_win_byts','init_bwd_win_byts','fwd_act_data_pkts','fwd_seg_size_min',
 'active_mean','active_std','active_max','active_min','idle_mean','idle_std',
 'idle_max','idle_min','cwr_flag_cnt','ece_flag_cnt', 'Label']



selected_columns2 = ['Total Fwd Packets', 'Total Backward Packets',
       'Fwd Packets Length Total', 'Bwd Packets Length Total',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 
       'CWE Flag Count', 'ECE Flag Count',
       'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size',
       'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 
       'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 
       'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets',
       'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes',
       'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean', 'Active Std',
       'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max',
       'Idle Min', 'Label']

len(selected_columns)

65

In [123]:
df2 = pd.read_csv('data/csv/cicddos_2019.csv')
df2 = df2[selected_columns2]

rename_columns = {
    'Total Fwd Packets': 'tot_fwd_pkts',
    'Total Backward Packets': 'tot_bwd_pkts',
    'Fwd Packets Length Total': 'totlen_fwd_pkts',
    'Bwd Packets Length Total': 'totlen_bwd_pkts',
    'Fwd Packet Length Max': 'fwd_pkt_len_max',
    'Fwd Packet Length Min': 'fwd_pkt_len_min',
    'Fwd Packet Length Mean': 'fwd_pkt_len_mean',
    'Fwd Packet Length Std': 'fwd_pkt_len_std',
    'Bwd Packet Length Max': 'bwd_pkt_len_max',
    'Bwd Packet Length Min': 'bwd_pkt_len_min',
    'Bwd Packet Length Mean': 'bwd_pkt_len_mean',
    'Bwd Packet Length Std': 'bwd_pkt_len_std',
    'Flow Bytes/s': 'flow_byts_s',
    'Flow Packets/s': 'flow_pkts_s',
    'Flow IAT Mean': 'flow_iat_mean',
    'Flow IAT Std':  'flow_iat_std',
    'Flow IAT Max': 'flow_iat_max',
    'Flow IAT Min': 'flow_iat_min',
    'Fwd IAT Total': 'fwd_iat_tot',
    'Fwd IAT Mean': 'fwd_iat_mean',
    'Fwd IAT Std': 'fwd_iat_std',
    'Fwd IAT Max': 'fwd_iat_max',
    'Fwd IAT Min': 'fwd_iat_min',
    'Bwd IAT Total': 'bwd_iat_tot',
    'Bwd IAT Mean': 'bwd_iat_mean',
    'Bwd IAT Std': 'bwd_iat_std',
    'Bwd IAT Max': 'bwd_iat_max',
    'Bwd IAT Min': 'bwd_iat_min',
    'Fwd PSH Flags': 'fwd_psh_flags',
    'Bwd PSH Flags': 'bwd_psh_flags',
    'Fwd URG Flags': 'fwd_urg_flags',
    'Bwd URG Flags': 'bwd_urg_flags',
    'Fwd Header Length': 'fwd_header_len',
    'Bwd Header Length': 'bwd_header_len',
    'Packet Length Min': 'pkt_len_min',
    'Packet Length Max': 'pkt_len_max',
    'Packet Length Mean': 'pkt_len_mean',
    'Packet Length Std': 'pkt_len_std',
    'Packet Length Variance': 'pkt_len_var',
    'Avg Packet Size': 'pkt_size_avg',
    'Avg Fwd Segment Size': 'fwd_seg_size_avg',
    'Avg Bwd Segment Size': 'bwd_seg_size_avg',
    'Fwd Avg Bytes/Bulk': 'fwd_byts_b_avg',
    'Fwd Avg Packets/Bulk': 'fwd_pkts_b_avg',
    'Bwd Avg Bytes/Bulk': 'bwd_byts_b_avg',
    'Bwd Avg Packets/Bulk': 'bwd_pkts_b_avg',
    'Subflow Fwd Packets': 'subflow_fwd_pkts',
    'Subflow Fwd Bytes': 'subflow_fwd_byts',
    'Subflow Bwd Packets': 'subflow_bwd_pkts',
    'Subflow Bwd Bytes': 'subflow_bwd_byts',
    'Init Fwd Win Bytes': 'init_fwd_win_byts',
    'Init Bwd Win Bytes': 'init_bwd_win_byts',
    'Fwd Act Data Packets': 'fwd_act_data_pkts',
    'Fwd Seg Size Min': 'fwd_seg_size_min',
    'Active Mean': 'active_mean',
    'Active Std': 'active_std',
    'Active Max': 'active_max',
    'Active Min': 'active_min',
    'Idle Mean': 'idle_mean',
    'Idle Std': 'idle_std',
    'Idle Max': 'idle_max',
    'Idle Min': 'idle_min',
    'CWE Flag Count': 'cwr_flag_cnt',
    'ECE Flag Count': 'ece_flag_cnt',
}


df2 = df2.rename(columns=rename_columns)

In [124]:
df2[df2["Label"] == "Syn"].sample(10)

,tot_fwd_pkts,tot_bwd_pkts,totlen_fwd_pkts,totlen_bwd_pkts,fwd_pkt_len_max,fwd_pkt_len_min,fwd_pkt_len_mean,fwd_pkt_len_std,bwd_pkt_len_max,bwd_pkt_len_min,...,fwd_seg_size_min,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,Label
29832,2,0,12.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,...,20,0.0,0.000000,0.0,0.0,0.000000e+00,0.000000,0.0,0.0,Syn
27511,2,0,12.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,...,20,0.0,0.000000,0.0,0.0,0.000000e+00,0.000000,0.0,0.0,Syn
29279,2,2,12.0,12.0,6.0,6.0,6.0,0.0,6.0,6.0,...,20,0.0,0.000000,0.0,0.0,0.000000e+00,0.000000,0.0,0.0,Syn
29447,8,2,48.0,12.0,6.0,6.0,6.0,0.0,6.0,6.0,...,20,34.0,57.157677,100.0,1.0,1.184862e+07,381301.434341,12258793.0,11504939.0,Syn
26848,2,0,12.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,...,20,0.0,0.000000,0.0,0.0,0.000000e+00,0.000000,0.0,0.0,Syn
25404,2,0,12.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,...,20,0.0,0.000000,0.0,0.0,0.000000e+00,0.000000,0.0,0.0,Syn
25167,2,0,12.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,...,20,0.0,0.000000,0.0,0.0,0.000000e+00,0.000000,0.0,0.0,Syn
29335,4,0,24.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,...,20,0.0,0.000000,0.0,0.0,0.000000e+00,0.000000,0.0,0.0,Syn
29909,2,0,12.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,...,20,0.0,0.000000,0.0,0.0,0.000000e+00,0.000000,0.0,0.0,Syn
25305,2,0,12.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,...,20,0.0,0.000000,0.0,0.0,0.000000e+00,0.000000,0.0,0.0,Syn


In [125]:

df = pd.read_csv('data/csv/test_syn_flood.csv')
df = df[selected_columns]

In [126]:
df[df["Label"] == "SYN"].sample(10)

,tot_fwd_pkts,tot_bwd_pkts,totlen_fwd_pkts,totlen_bwd_pkts,fwd_pkt_len_max,fwd_pkt_len_min,fwd_pkt_len_mean,fwd_pkt_len_std,bwd_pkt_len_max,bwd_pkt_len_min,...,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,cwr_flag_cnt,ece_flag_cnt,Label
5,1,0,60,0,60.0,60.0,60.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,SYN
2,1,0,85,0,85.0,85.0,85.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,SYN
10,1,0,87,0,87.0,87.0,87.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,SYN
8,1,0,87,0,87.0,87.0,87.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,SYN
6,1,0,85,0,85.0,85.0,85.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,SYN
0,1,0,85,0,85.0,85.0,85.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,SYN
4,1,0,85,0,85.0,85.0,85.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,SYN
7,1,0,60,0,60.0,60.0,60.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,SYN
1,1,0,60,0,60.0,60.0,60.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,SYN
3,1,0,60,0,60.0,60.0,60.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,SYN


In [127]:

datadir = 'test'
def convert(df_normalized_splited, label, num):
    # Kích thước ảnh ban đầu (5x15) => Mở rộng mỗi điểm thành 3x3 pixel => Ảnh mới (15x45)
    image_size = (8, 8) # kích thước ảnh ban đầu
    upscale_factor = 28 # tỉ lệ tăng kích thước điểm ảnh => size ảnh: (8x28) x (8x28)
    new_image_size = (image_size[0] * upscale_factor, image_size[1] * upscale_factor)

    os.makedirs(f"data/{datadir}/{label}", exist_ok=True)  # Tạo thư mục nếu chưa có

    i = 1
    for row in df_normalized_splited.values:
        # Chuyển đổi dòng thành ma trận ảnh ban đầu (8x8)
        image_array = np.array(row).reshape(image_size)
        image_array = np.nan_to_num(image_array)  # Thay thế giá trị NaN bằng 0

        # Phóng to mỗi pixel np.kron()
        upscale_matrix = np.ones((upscale_factor, upscale_factor))
        enlarged_image_array = np.kron(image_array, upscale_matrix)

        # Chuyển đổi sang ảnh
        image = Image.fromarray((enlarged_image_array * 255).astype(np.uint8))  # Chuyển sang RGB
        image = image.convert("RGB")

        
        ###### thêm nhiễu vào ảnh ######

        # Xoay ảnh ngẫu nhiên (-15° đến 15°)
        rotate_prob = random.random() < 0.8
        if rotate_prob:  
            angle = random.uniform(-90, 90)  
            image = image.rotate(angle)


        # Lật ảnh ngẫu nhiên
        flip_prob = random.random() < 0.8
        if flip_prob:
            if random.random() < 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)  # Lật ngang
            else:
                image = image.transpose(Image.FLIP_TOP_BOTTOM)  # Lật dọc

        # Điều chỉnh độ sáng ngẫu nhiên (từ 70% đến 130%)
        brightness_prob = random.random() < 0.8
        if brightness_prob:
            enhancer = ImageEnhance.Brightness(image)
            factor = random.uniform(0.7, 1.8)  # Thay đổi độ sáng từ 70% đến 130%
            image = enhancer.enhance(factor)


        # Xác suất ngẫu nhiên để thêm hiệu ứng (30% ảnh bị làm mờ, 30% ảnh bị nhiễu)
        blur_prob = random.random() <= 0.5  # 30% xác suất làm mờ
        noise_prob = random.random() <= 0.5  # 30% xác suất thêm nhiễu

        '''
        chỉ làm mờ: 0.3, không mờ 0,7 ==> 0.21 xác xuất chỉ mờ
        chỉ làm nhiễu: 0.3, không nhiễu 0,7 ==> 0.21 xác xuất chỉ nhiễu
        vừa mờ: 0.3, vừa nhiễu 0.3 ==> xác xuất vừa mờ vừa nhiễu: 0.09
        không mờ: 0.7, không nhiễu: 0.7 ==> xác xuất không mờ không nhiễu: 0.49
        '''

        # làm mờ ảnh
        if blur_prob:
            image = image.filter(ImageFilter.GaussianBlur(radius=random.uniform(10, 20))) # mức độ mờ từ 1 đến 5

        #làm nhiễu ảnh
        if noise_prob:
            noise = np.random.normal(0, 50, (new_image_size[0], new_image_size[1]))  # Thêm nhiễu Gaussian
            noisy_image_array = np.array(image.convert("L")) + noise  # Chuyển sang grayscale trước khi thêm nhiễu
            noisy_image_array = np.clip(noisy_image_array, 0, 255).astype(np.uint8)  # Giữ giá trị trong khoảng 0-255
            image = Image.fromarray(noisy_image_array).convert("RGB")

        # Lưu ảnh
        image.save(f"data/{datadir}/{label}/{str(i)}.png")

        i += 1
        if i > num:
            break

def setup_to_convert(df_normalized, label):
    os.makedirs(f'data/{datadir}/{label}', exist_ok=True)

    # tổng số lượng ảnh train + valid + test của mỗi nhãn
    n = 200
    convert(df_normalized, label, num=n)

In [128]:
# Nhóm dữ liệu theo cột 'Label'
grouped = df.groupby('Label')

# Tạo dictionary để lưu các DataFrame tương ứng với từng nhãn
dfs = {label: group for label, group in grouped}

def process_label(label, df_label):
    df_drop_label = df_label.drop(columns=['Label'])
    
    df_drop_label.replace([np.inf, -np.inf], np.nan, inplace=True)  # Thay giá trị vô hạn bằng NaN
    df_drop_label.fillna(df_drop_label.median(), inplace=True)  # Điền giá trị NaN bằng giá trị trung vị

    data_features = np.log1p(df_drop_label + 1)
    data_standardized = (data_features - data_features.mean()) / data_features.std()
    data_normalized = data_standardized

    setup_to_convert(data_normalized, label)

# Sử dụng ThreadPoolExecutor để chạy đa luồng với tối đa 5 luồng
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(process_label, label, df_label) for label, df_label in dfs.items()]

    # Đợi tất cả các task hoàn thành
    for future in futures:
        future.result()

print("Hoàn thành xử lý đa luồng!")

Hoàn thành xử lý đa luồng!
